# 28. Train a multilayer neural network with PyTorch

**Level:** Intermediate

**Prerequisites:** 27 and 11: autograd and data splits.

[Course index](00_START_HERE.ipynb) | [Previous](27_Neurons_Backpropagation_and_Autograd.ipynb) | [Next](29_Convolutional_Neural_Networks.ipynb)

**Topics:** PyTorch, MLP, Training loop, Optimizer, Cross-entropy, Inference mode.

**Matching script:** `examples\28_Training_a_PyTorch_MLP.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

An MLP composes affine layers and nonlinear activations. Its output for classification is often
raw logits. CrossEntropyLoss expects logits and integer class indices; applying softmax first
is unnecessary and can harm numerical behavior. Use softmax later when inspecting probabilities.

A training step is: forward pass, loss calculation, gradient reset, backward pass, optimizer
update. Multiple epochs reuse training examples. Validation monitors model choice; test data
must not influence training or hyperparameter selection. Batch size and learning rate are
hyperparameters, while weight matrices and biases are learned parameters.

model.train() enables training behavior for layers such as dropout. model.eval() uses inference
behavior for such layers. torch.inference_mode reduces gradient bookkeeping but is not a
substitute for eval. Use both deliberately when evaluating a model with those layers.

## Mathematics or data contract

For class c, cross-entropy is -log(softmax(logits)[c]); optimize mean loss over training examples.

## Important limitation

The full-batch two-moons model is small CPU practice, not a production training recipe.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Actually train and test a small nonlinear classifier

**Purpose:** Run a real CPU-sized training loop with held-out evaluation.

### Read the code in this order
1. Split synthetic data and fit scaling on training only.
2. Create a two-layer network and optimizer.
3. Train with integer targets and raw logits.
4. Compare initial/final loss and assert useful held-out accuracy.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `transform(...)` applies already-fitted preprocessing; it does not refit it.
- `backward()` computes/accumulates gradients through the recorded computation graph.
- `zero_grad()` clears earlier gradients before the next independent training update.
- `optimizer.step()` updates only the parameters owned by that optimizer.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.
- `with`/`async with` bounds a resource lifetime or a temporary execution mode.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
training loss: 0.622 -> 0.007
held-out accuracy: 1.0
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.set_num_threads(1)
torch.manual_seed(42)
X, y = make_moons(n_samples=260, noise=0.12, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
train_X = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
test_X = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
train_y = torch.tensor(y_train, dtype=torch.long)
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
optimizer, criterion = torch.optim.Adam(model.parameters(), lr=0.03), nn.CrossEntropyLoss()
initial = criterion(model(train_X), train_y).item()
model.train()
for _ in range(140):
    optimizer.zero_grad()
    loss = criterion(model(train_X), train_y)
    loss.backward()
    optimizer.step()
model.eval()
with torch.inference_mode():
    accuracy = float((model(test_X).argmax(1).numpy() == y_test).mean())
    final = criterion(model(train_X), train_y).item()
assert final < initial and accuracy > 0.85
print("training loss:", round(initial, 3), "->", round(final, 3))
print("held-out accuracy:", round(accuracy, 3))

training loss: 0.622 -> 0.007
held-out accuracy: 1.0


### Interpret example 1

The result should demonstrate this purpose: **Run a real CPU-sized training loop with held-out evaluation.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add a validation partition and early stopping without checking the test set every epoch.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `PyTorch` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: For class c, cross-entropy is -log(softmax(logits)[c]); optimize mean loss over training examples.
- Name the boundary of the demonstration: The full-batch two-moons model is small CPU practice, not a production training recipe.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://docs.pytorch.org/tutorials/beginner/basics/intro.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.